In [1]:
import pandas as pd

# Entendimento do Problema
- A farmácia precisa entender consumo, estoque e risco de ruptura dos produtos.
- Objetivo é analisar consumo por produto, grupo e setor, identificar itens críticos e gerar indicadores para apoiar a reposição.
- Começaremos analisando a base de estoque, posteriormente a de consumo, para fazer o merge.
- Ultilizarei sempre o conceito de ETL.

## Visualização Inicial DF Estoque

In [2]:
df_estoque = pd.read_csv("../dados/estoque_atual.csv", sep=",")
display(df_estoque.head(4))
display(df_estoque.tail(4))

,Codigo,Produto,Grupo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
0,1,Dipirona 500mg/mL,Medicamento,122,122,592,13
1,2,"Soro Fisiologico 0,9% 500mL",Solucoes,91,50,864,13
2,3,Soro Glicosado 5% 500mL,Solucoes,478,117,689,6
3,4,Equipo Macrogotas,Material Hospitalar,891,181,511,4


,Codigo,Produto,Grupo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
21,22,Mascara Cirurgica,Material Hospitalar,84,118,838,3
22,23,Compressa Gaze,Material Hospitalar,667,92,430,19
23,24,Esparadrapo,Material Hospitalar,411,192,810,3
24,25,Alcool 70%,Material Hospitalar,398,34,723,9


In [3]:
display(df_estoque.info())
display(df_estoque.describe())
display(df_estoque.columns.to_list())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Codigo                25 non-null     int64 
 1   Produto               25 non-null     object
 2   Grupo                 25 non-null     object
 3   Estoque_Atual         25 non-null     int64 
 4   Estoque_Minimo        25 non-null     int64 
 5   Estoque_Maximo        25 non-null     int64 
 6   Tempo_Reposicao_Dias  25 non-null     int64 
dtypes: int64(5), object(2)
memory usage: 1.5+ KB


None

,Codigo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
count,25.000000,25.000000,25.000000,25.000000,25.000000
mean,13.000000,435.080000,96.840000,650.840000,10.840000
std,7.359801,271.175976,54.558592,206.642743,5.080026
min,1.000000,60.000000,31.000000,271.000000,3.000000
25%,7.000000,186.000000,47.000000,511.000000,6.000000
50%,13.000000,411.000000,83.000000,687.000000,12.000000
75%,19.000000,504.000000,121.000000,810.000000,15.000000
max,25.000000,977.000000,193.000000,993.000000,19.000000


['Codigo',
 'Produto',
 'Grupo',
 'Estoque_Atual',
 'Estoque_Minimo',
 'Estoque_Maximo',
 'Tempo_Reposicao_Dias']

## Verificação de Nulo e Duplicados DF Estoque

In [4]:
display(df_estoque.isna().sum())
display(df_estoque.duplicated().sum())

Codigo                  0
Produto                 0
Grupo                   0
Estoque_Atual           0
Estoque_Minimo          0
Estoque_Maximo          0
Tempo_Reposicao_Dias    0
dtype: int64

np.int64(0)

In [5]:
display(df_estoque["Grupo"].unique())

array(['Medicamento', 'Solucoes', 'Material Hospitalar', 'Antibiotico'],
      dtype=object)

## Análise Inicial da Base DF Estoque

- Quantos produtos e grupos existem?

In [6]:
display(df_estoque["Produto"].unique())
display(df_estoque["Grupo"].unique())

array(['Dipirona 500mg/mL', 'Soro Fisiologico 0,9% 500mL',
       'Soro Glicosado 5% 500mL', 'Equipo Macrogotas', 'Seringa 10mL',
       'Seringa 20mL', 'Agulha 25x7', 'Luva Procedimento M',
       'Omeprazol 40mg', 'Ceftriaxona 1g', 'Ampicilina 1g',
       'Metoclopramida 10mg', 'Ondansetrona 4mg', 'Furosemida 20mg',
       'Hidrocortisona 100mg', 'Adrenalina 1mg/mL', 'Atropina 0,25mg/mL',
       'Cloreto de Sodio 20%', 'Glicose 50%', 'Cateter Venoso 20G',
       'Cateter Venoso 22G', 'Mascara Cirurgica', 'Compressa Gaze',
       'Esparadrapo', 'Alcool 70%'], dtype=object)

array(['Medicamento', 'Solucoes', 'Material Hospitalar', 'Antibiotico'],
      dtype=object)

- Qual grupo tem mais produtos?

In [21]:
df_produto_grupo = df_estoque.groupby("Grupo").agg(
    Total_Produtos=("Produto", "count")
).sort_values(by="Total_Produtos", ascending=False).reset_index()
display(df_produto_grupo)

,Grupo,Total_Produtos
0,Material Hospitalar,11
1,Medicamento,8
2,Solucoes,4
3,Antibiotico,2


- Quantos produtos estão abaixo do estoque mínimo?

In [10]:
condicao_estoque_abaixo = df_estoque["Estoque_Atual"] < df_estoque["Estoque_Minimo"]

display(df_estoque[condicao_estoque_abaixo])

,Codigo,Produto,Grupo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
14,15,Hidrocortisona 100mg,Medicamento,100,193,954,10
20,21,Cateter Venoso 22G,Material Hospitalar,60,186,400,15
21,22,Mascara Cirurgica,Material Hospitalar,84,118,838,3


- Quantos produtos estão acima do estoque máximo?

In [12]:
condicao_estoque_acima = df_estoque["Estoque_Atual"] > df_estoque["Estoque_Maximo"]
display(df_estoque[condicao_estoque_acima])

,Codigo,Produto,Grupo,Estoque_Atual,Estoque_Minimo,Estoque_Maximo,Tempo_Reposicao_Dias
3,4,Equipo Macrogotas,Material Hospitalar,891,181,511,4
5,6,Seringa 20mL,Material Hospitalar,479,87,308,14
6,7,Agulha 25x7,Material Hospitalar,876,78,752,12
7,8,Luva Procedimento M,Material Hospitalar,495,44,433,17
8,9,Omeprazol 40mg,Medicamento,977,80,846,5
13,14,Furosemida 20mg,Medicamento,447,37,271,16
16,17,"Atropina 0,25mg/mL",Medicamento,791,83,759,14
19,20,Cateter Venoso 20G,Material Hospitalar,768,111,363,15
22,23,Compressa Gaze,Material Hospitalar,667,92,430,19


- Qual o estoque total por grupo?

In [20]:
df_estoque_grupo = df_estoque.groupby(["Grupo"]).agg(
    Estoque_Total = ("Estoque_Atual","sum")
).sort_values(by="Estoque_Total", ascending=False).reset_index()
display(df_estoque_grupo)

,Grupo,Estoque_Total
0,Material Hospitalar,5492
1,Medicamento,3464
2,Solucoes,1231
3,Antibiotico,690


- Quais produtos têm menor estoque atual?

In [23]:
df_estoque_produto = df_estoque.groupby(["Grupo", "Produto","Estoque_Minimo"]).agg(
    Estoque_Produto_Total=("Estoque_Atual","sum")
).sort_values(by="Estoque_Produto_Total").reset_index()
display(df_estoque_produto.head(10))

,Grupo,Produto,Estoque_Minimo,Estoque_Produto_Total
0,Material Hospitalar,Cateter Venoso 22G,186,60
1,Material Hospitalar,Mascara Cirurgica,118,84
2,Solucoes,"Soro Fisiologico 0,9% 500mL",50,91
3,Medicamento,Hidrocortisona 100mg,193,100
4,Medicamento,Dipirona 500mg/mL,122,122
5,Solucoes,Cloreto de Sodio 20%,43,181
6,Antibiotico,Ampicilina 1g,47,186
7,Medicamento,Metoclopramida 10mg,38,261
8,Medicamento,Ondansetrona 4mg,121,359
9,Material Hospitalar,Seringa 10mL,187,363


## Visualização Inicial DF Consumo Mensal

In [24]:
df_consumo_mensal = pd.read_csv("../dados/consumo_mensal.csv", sep=",")
display(df_consumo_mensal.head())
display(df_consumo_mensal.tail())

,Data,Codigo,Produto,Grupo,Setor,Quantidade_Consumida
0,2024-01-01,14,Furosemida 20mg,Medicamento,Urgencia,14
1,2024-01-01,20,Cateter Venoso 20G,Material Hospitalar,UTI,18
2,2024-01-01,4,Equipo Macrogotas,Material Hospitalar,Bloco Cirurgico,99
3,2024-01-01,10,Ceftriaxona 1g,Antibiotico,Urgencia,13
4,2024-01-01,23,Compressa Gaze,Material Hospitalar,Urgencia,105


,Data,Codigo,Produto,Grupo,Setor,Quantidade_Consumida
2268,2024-06-30,7,Agulha 25x7,Material Hospitalar,UTI,86
2269,2024-06-30,23,Compressa Gaze,Material Hospitalar,Urgencia,75
2270,2024-06-30,13,Ondansetrona 4mg,Medicamento,UTI,22
2271,2024-06-30,22,Mascara Cirurgica,Material Hospitalar,Urgencia,82
2272,2024-06-30,6,Seringa 20mL,Material Hospitalar,Clínica Médica,93


In [25]:
df_consumo_mensal.info()
df_consumo_mensal.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2273 entries, 0 to 2272
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Data                  2273 non-null   object
 1   Codigo                2273 non-null   int64 
 2   Produto               2273 non-null   object
 3   Grupo                 2273 non-null   object
 4   Setor                 2273 non-null   object
 5   Quantidade_Consumida  2273 non-null   int64 
dtypes: int64(2), object(4)
memory usage: 106.7+ KB


,Codigo,Quantidade_Consumida
count,2273.000000,2273.000000
mean,13.128905,45.451826
std,7.202529,30.089340
min,1.000000,1.000000
25%,7.000000,22.000000
50%,13.000000,39.000000
75%,19.000000,64.000000
max,25.000000,119.000000


## Tratamento da Coluna Data do DF Consumo Mensal

- Objetivo: transformar a coluna para o formato datetime64 e criar colunas auxiliares como mês.

In [29]:
df_consumo_mensal["Data"] = pd.to_datetime(df_consumo_mensal["Data"], format="%Y-%m-%d")
df_consumo_mensal["Mes"] = df_consumo_mensal["Data"].dt.month_name()
display(df_consumo_mensal.head())
df_consumo_mensal.info()

,Data,Codigo,Produto,Grupo,Setor,Quantidade_Consumida,Mes
0,2024-01-01,14,Furosemida 20mg,Medicamento,Urgencia,14,January
1,2024-01-01,20,Cateter Venoso 20G,Material Hospitalar,UTI,18,January
2,2024-01-01,4,Equipo Macrogotas,Material Hospitalar,Bloco Cirurgico,99,January
3,2024-01-01,10,Ceftriaxona 1g,Antibiotico,Urgencia,13,January
4,2024-01-01,23,Compressa Gaze,Material Hospitalar,Urgencia,105,January


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2273 entries, 0 to 2272
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Data                  2273 non-null   datetime64[ns]
 1   Codigo                2273 non-null   int64         
 2   Produto               2273 non-null   object        
 3   Grupo                 2273 non-null   object        
 4   Setor                 2273 non-null   object        
 5   Quantidade_Consumida  2273 non-null   int64         
 6   Mes                   2273 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(4)
memory usage: 124.4+ KB


## Verificação de Nulos e Duplicados DF Consumo Mensal

In [32]:
display(df_consumo_mensal.isna().sum())
display(df_consumo_mensal.duplicated().sum())

Data                    0
Codigo                  0
Produto                 0
Grupo                   0
Setor                   0
Quantidade_Consumida    0
Mes                     0
dtype: int64

np.int64(0)

## Validação da Quantidade Consumida DF Consumo Mensal

- Regra básica:

Quantidade consumida deve ser maior que zero

In [33]:
condicao_consumo = df_consumo_mensal["Quantidade_Consumida"] < 0
display(df_consumo_mensal[condicao_consumo])

,Data,Codigo,Produto,Grupo,Setor,Quantidade_Consumida,Mes


## Padronização de Setor DF Estoque

- Verificar se há valores únicos de Setor

In [34]:
setores_unicos = df_consumo_mensal["Setor"].unique()
display(setores_unicos)

array(['Urgencia', 'UTI', 'Bloco Cirurgico', 'Clínica Médica'],
      dtype=object)